This is a test to try and use python to predict the variac voltages. It probably won't work!

In [7]:
import pandas as pd
from simple_pid import PID

# Load data from CSV
def load_data_from_csv(file_path):
    df = pd.read_csv(file_path)
    return df

# Function to run the PID control loop
def run_pid_control(data, setpoints, Kp, Ki, Kd):
    pids = []
    for i in range(len(setpoints)):
        pid = PID(Kp[i], Ki[i], Kd[i], setpoint=setpoints[i])
        pid.sample_time = 1  # sampling time in seconds
        pids.append(pid)

    control_voltages = []
    for index, row in data.iterrows():
        temperatures = [row[f'T{i}'] for i in range(1, 12)]
        control_values = []
        for i, pid in enumerate(pids):
            control_value = pid(temperatures[i])
            control_values.append(control_value)

        control_voltages.append(control_values)

        print(f"Time: {row['Time']}, Temperatures: {temperatures}, Control Voltages: {control_values}")

    return control_voltages

# Load data from CSV
file_path = 'bakeoutData.csv'
df = load_data_from_csv(file_path)

# PID parameters
setpoints = [100]*11  # Desired temperatures
Kp = [2.0]*11           # Proportional gains
Ki = [0.1]*11           # Integral gains
Kd = [0.5]*11           # Derivative gains

# Run the PID control loop
voltages = run_pid_control(df, setpoints, Kp, Ki, Kd)

KeyError: 'T1'

In [8]:
df.head()

,Time,Pressure,Variac (v),Unnamed: 3,Unnamed: 4,Unnamed: 5,Temperatures,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Notes
0,NaN,torr,E/W,Bot,Top,RGA,T1 tower gate,T2 Telescope South Top,T3 MOT core,T4 Ion Pump east,T5 1.33 gate valve,T6 Telescope North Bot,T7 1.33 window NorthEast,T8 RGA cold flange,T9 turbo pump angle valve,T10 Top window,11 Ion pump West,Average temp,NaN
1,12:00,NaN,20,8,24,20,32.43,NaN,42.84,31.35,27.56,NaN,34.9,32.81,NaN,29.67,33.52,33.135,NaN
2,12:20,2.00E-07,30,15,35,30,39.99,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,39.99,NaN
3,12:44,1.20E-06,30,15,35,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,13:07,2.40E-06,35,20,40,35,49.11,NaN,63.88,47.78,37.58,NaN,61.42,44.15,NaN,51.05,47.91,50.36,NaN


In [9]:
import pandas as pd
from simple_pid import PID

# Load data from CSV
def load_data_from_csv(file_path):
    df = pd.read_csv(file_path)
    # Remove the top row
    df = df.iloc[1:]
    # Remove the 'Unnamed' columns
    df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
    # Rename the temperature columns
    temperature_columns = df.columns[6:-1]
    new_temperature_columns = [f'T{i}' for i in range(1, len(temperature_columns) + 1)]
    df.columns = df.columns[:6] + new_temperature_columns + [df.columns[-1]]
    return df

# Function to run the PID control loop
def run_pid_control(data, setpoints, Kp, Ki, Kd):
    pids = []
    for i in range(len(setpoints)):
        pid = PID(Kp[i], Ki[i], Kd[i], setpoint=setpoints[i])
        pid.sample_time = 1  # sampling time in seconds
        pids.append(pid)

    control_voltages = []
    for index, row in data.iterrows():
        temperatures = [row[f'T{i}'] for i in range(1, 12)]
        control_values = []
        for i, pid in enumerate(pids):
            control_value = pid(temperatures[i-1]) # Corrected index
            control_values.append(control_value)

        control_voltages.append(control_values)

        print(f"Time: {row['Time']}, Temperatures: {temperatures}, Control Voltages: {control_values}")

    return control_voltages

# Load data from CSV
file_path = 'bakeoutData.csv'
df = load_data_from_csv(file_path)

# PID parameters
setpoints = [100]*11  # Desired temperatures
Kp = [2.0]*11           # Proportional gains
Ki = [0.1]*11           # Integral gains
Kd = [0.5]*11           # Derivative gains

# Run the PID control loop
voltages = run_pid_control(df, setpoints, Kp, Ki, Kd)

ValueError: operands could not be broadcast together with shapes (5,) (0,) 

In [10]:
df

,Time,Pressure,Variac (v),Unnamed: 3,Unnamed: 4,Unnamed: 5,Temperatures,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Notes
0,NaN,torr,E/W,Bot,Top,RGA,T1 tower gate,T2 Telescope South Top,T3 MOT core,T4 Ion Pump east,T5 1.33 gate valve,T6 Telescope North Bot,T7 1.33 window NorthEast,T8 RGA cold flange,T9 turbo pump angle valve,T10 Top window,11 Ion pump West,Average temp,NaN
1,12:00,NaN,20,8,24,20,32.43,NaN,42.84,31.35,27.56,NaN,34.9,32.81,NaN,29.67,33.52,33.135,NaN
2,12:20,2.00E-07,30,15,35,30,39.99,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,39.99,NaN
3,12:44,1.20E-06,30,15,35,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,13:07,2.40E-06,35,20,40,35,49.11,NaN,63.88,47.78,37.58,NaN,61.42,44.15,NaN,51.05,47.91,50.36,NaN
5,13:22,4.47E-06,35,20,40,35,54.68,NaN,74.12,53.1,40.75,NaN,70.94,46.9,NaN,56.88,52.59,56.245,NaN
6,13:42,7.45E-06,35,20,40,35,63,NaN,81.15,60.26,45.38,NaN,80.52,50.46,NaN,65.03,59,63.1,NaN
7,13:50,8.52E-06,30,15,35,30,66.38,NaN,83.57,62.96,47.16,NaN,83.45,51.7,NaN,68.17,61.59,65.6225,NaN
8,14:17,7.88E-06,30,15,35,30,72.89,NaN,83,66.85,50.11,NaN,75.69,52.35,NaN,72.54,65.31,67.3425,NaN
9,14:50,8.00E-06,30,15,35,30,78.86,NaN,85.37,71.53,52.21,NaN,72.33,52.46,NaN,76.73,69.46,69.86875,NaN
